# Configurable GPS Acquisition + Tracking (L1CA or L2C)

This notebook demonstrates a unified acquisition/tracking flow using `SignalDefinition` and `CorrelatorStrategy` from `utils.signal_interfaces`.

Set `SIGNAL_FAMILY` in the config cell to switch between GPS L1 C/A and GPS L2C (CM/CL interleaved tracking).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import utils
from utils import bpsk_acquisition, collect_metadata_utils, sample_streaming, tracking_bpsk_aligned
from utils.signal_interfaces import (
    SignalFamily,
    build_signal_definitions,
    build_acquisition_code_params,
    create_tracking_channels,
)

plt.rcParams.update({"font.size": 14})

In [ ]:
# ----- User configuration -----
SIGNAL_FAMILY = SignalFamily.L2C  # SignalFamily.L1CA or SignalFamily.L2C
EXPERIMENT_INDEX = 2

# Acquisition settings
L1_ACQ_REPLICA_MS = 4
L1_ACQ_NUM_BLOCKS = 8
L2_ACQ_REPLICA_MS = 20
L2_ACQ_NUM_BLOCKS = 1

# Tracking settings
BUFFER_DURATION_MS = 200
BLOCK_DURATION_MS = 1
TRACK_DURATION_MS = 2000
START_TRACKING_IN_PLL_MODE = False

print(f"Signal family: {SIGNAL_FAMILY.value}")

In [ ]:
local_data_dir = Path(utils.__file__).parent.parent / "local-data"
collects_dir = local_data_dir / "collects"
available_experiment_names = sorted(fp.name for fp in collects_dir.iterdir())
print("Available experiments:", ", \
,
,
,
,
metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

target_band = "L1" if SIGNAL_FAMILY == SignalFamily.L1CA else "L2"
candidate_collect_ids = []
for cid in metadata.collect_ids:
    collect_cfg = metadata.collects[cid]
    channel_cfg = metadata.channel_configurations[collect_cfg.channel_config_id]
    if target_band in channel_cfg.bands:
        candidate_collect_ids.append(cid)

if len(candidate_collect_ids) == 0:
    raise RuntimeError(f"No collect found for target band {target_band}.")

collect_id = candidate_collect_ids[0]
band_id = target_band
collect_config = metadata.collects[collect_id]
channel_config = metadata.channel_configurations[collect_config.channel_config_id]
band_config = metadata.band_configurations[band_id]

samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
inter_freq_hz = band_config.inter_freq
collect_filepath = experiment_dir / collect_config.filename

print(f"Selected collect_id={collect_id}, band_id={band_id}")
print(f"Collect filepath: {collect_filepath}")
print(f"Sample rate: {samp_rate} Hz")

In [ ]:
signal_definitions = build_signal_definitions(SIGNAL_FAMILY)
acq_code_params = build_acquisition_code_params(signal_definitions)

acq_buffer_duration_ms = 40
acq_buffer_size_samples = int(samp_rate * acq_buffer_duration_ms / 1e3)
byte_buffer = bytearray(
    sample_streaming.compute_sample_array_size_bytes(
        acq_buffer_size_samples, sample_params.bit_depth, sample_params.is_complex
    )
)
samples = np.zeros(acq_buffer_size_samples, dtype=np.complex64)
baseband_samples = np.zeros(acq_buffer_size_samples, dtype=np.complex64)

with open(collect_filepath, "rb") as f:
    f.readinto(byte_buffer)

sample_streaming.convert_to_complex64_samples(byte_buffer, samples, sample_params)
sample_streaming.mixdown_samples(
    samples,
    baseband_samples,
    samp_rate,
    initial_phase_cycles=0.0,
    freq_hz=inter_freq_hz,
)
baseband_samples -= np.mean(baseband_samples)

if SIGNAL_FAMILY == SignalFamily.L1CA:
    replica_duration_ms = L1_ACQ_REPLICA_MS
    num_blocks = L1_ACQ_NUM_BLOCKS
    p_fa_total = 1e-9
else:
    replica_duration_ms = L2_ACQ_REPLICA_MS
    num_blocks = L2_ACQ_NUM_BLOCKS
    p_fa_total = 1e-6

acq_config = bpsk_acquisition.AcquisitionConfiguration(
    replica_duration_ms=replica_duration_ms,
    num_blocks=num_blocks,
    sample_rate=samp_rate,
    min_search_doppler_hz=-5000,
    max_search_doppler_hz=5000,
)
num_detection_tests = acq_config.num_doppler_bins * acq_config.replica_length_samples
p_fa_bin = 1 - (1 - p_fa_total) ** (1 / num_detection_tests)

acq_results = bpsk_acquisition.run_acquisition(
    sample_block=baseband_samples,
    sample_block_uptime_epoch_ms=0.0,
    acq_config=acq_config,
    code_parameters=acq_code_params,
    prob_false_alaram=p_fa_bin,
    print_progress=True,
    noise_var_method="abscorrvar",
)

acquired_signal_ids = sorted([
    sid for sid, result in acq_results.items() if result.signal_detected
])
print(f"Acquired signals ({SIGNAL_FAMILY.value}): {', '.join(acquired_signal_ids)}")

In [ ]:
if len(acquired_signal_ids) == 0:
    raise RuntimeError("No signals acquired; cannot start tracking.")

buffer_size_samples = int(samp_rate * BUFFER_DURATION_MS / 1e3)
num_buffers_to_process = TRACK_DURATION_MS // BUFFER_DURATION_MS
output_capacity = TRACK_DURATION_MS // BLOCK_DURATION_MS

tracking_loop_params = tracking_bpsk_aligned.TrackingLoopParameters(
    DLL_bandwidth_hz=2.0,
    PLL_bandwidth_hz=20.0,
    FLL_bandwidth_hz=50.0,
    nominal_update_period_ms=BLOCK_DURATION_MS,
    corr_period_ms=BLOCK_DURATION_MS,
    EPL_chip_spacing=0.5,
    prompt_corr_circ_length_threshold=0.9,
)

tracking_channels = create_tracking_channels(
    signal_definitions=signal_definitions,
    acquisition_results=acq_results,
    tracking_signal_ids=acquired_signal_ids,
    loop_params=tracking_loop_params,
    output_capacity=output_capacity,
    start_mode_pll=START_TRACKING_IN_PLL_MODE,
)

with sample_streaming.FileSampleStream(
    collect_filepath,
    sample_params,
    buffer_size_samples,
) as sample_stream:
    sample_buffer_generator = sample_stream.sample_buffer_generator()
    for i_buffer, buffer_samples in enumerate(sample_buffer_generator):
        if i_buffer >= num_buffers_to_process:
            break
        uptime_ms = i_buffer * BUFFER_DURATION_MS
        mixdown_phase_cycles = inter_freq_hz * (uptime_ms * 1e-3)
        sample_streaming.mixdown_samples(
            buffer_samples,
            buffer_samples,
            samp_rate,
            initial_phase_cycles=mixdown_phase_cycles,
            freq_hz=inter_freq_hz,
        )

        sample_buffer = sample_streaming.SampleBuffer(
            samples=buffer_samples,
            start_uptime_ms=uptime_ms,
            samp_rate=samp_rate,
        )
        for adapter in tracking_channels.values():
            adapter.process_sample_buffer(sample_buffer)

print("Tracking complete")

In [ ]:
plot_sig_id = acquired_signal_ids[0]
adapter = tracking_channels[plot_sig_id]
outputs = adapter.outputs

plot_time = outputs.uptime_epoch_ms * 1e-3
prompt0 = adapter.get_prompt_component(component=0)

fig = plt.figure(figsize=(12, 8), dpi=150)
axes = fig.subplots(2, 1, sharex=True)

axes[0].scatter(plot_time, prompt0.real, s=2, color="tab:red", label="Prompt I")
axes[0].scatter(plot_time, prompt0.imag, s=2, color="tab:blue", label="Prompt Q")
axes[0].set_ylabel("Prompt Corr")
axes[0].set_title(f"{SIGNAL_FAMILY.value.upper()} Tracking: {plot_sig_id}")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(plot_time, outputs.doppler_freq_hz, lw=1.5, color="tab:green")
axes[1].set_ylabel("Doppler [Hz]")
axes[1].set_xlabel("Uptime [s]")
axes[1].grid(True)

if SIGNAL_FAMILY == SignalFamily.L2C:
    prompt1 = adapter.get_prompt_component(component=1)
    fig2 = plt.figure(figsize=(12, 4), dpi=150)
    ax2 = fig2.add_subplot(1, 1, 1)
    ax2.scatter(plot_time, np.abs(prompt0), s=2, label="CM |Prompt|", color="tab:purple")
    ax2.scatter(plot_time, np.abs(prompt1), s=2, label="CL |Prompt|", color="tab:orange")
    ax2.set_title(f"L2C Components: {plot_sig_id}")
    ax2.set_ylabel("Magnitude")
    ax2.set_xlabel("Uptime [s]")
    ax2.grid(True)
    ax2.legend()

plt.show()